# Emergency canonical ZINC/QM9 worker

This one notebook controls the fixed 30-checkpoint corpus. Put `zinc_qm9_best_checkpoints.tar` and `zinc_qm9_best_checkpoints.tar.sha256` directly in `DRIVE_FOLDER`. Run `MODE=setup` once: it verifies/extracts the corpus, warms the shared ZINC and QM9 datasets under an exclusive Drive lock, and creates 30 commit-pinned notebooks under `worker_notebooks/`. Open as many of those generated notebooks as Colab permits and **Run all**; each has a different `WORKER_INDEX` and an isolated `task/seed` output.

`preflight` verifies one selection without model work. `smoke` runs the selected checkpoint against a separate three-graph cache. `status` prints the 30-worker matrix. Run `finalize` only after all rows are validated. Production starts with all 48 graphs on an A100-80GB and halves a failing CUDA batch automatically. Do not launch the same worker index twice.


In [ ]:
# Controls: generated worker copies change only MODE, WORKER_INDEX, and the pinned revision.
MODE = "setup"  # @param ["setup", "preflight", "smoke", "worker", "status", "finalize"]
WORKER_INDEX = 0  # @param {type:"integer"}
DRIVE_FOLDER = "/content/drive/MyDrive/graph_specialisation_metrics/multi_seed_models"

# Optional explicit selector: set both to override WORKER_INDEX.
TASK = ""
TRAIN_SEED = -1
# Zero selects the GPU-aware profile: A100/H100 >=75 GiB -> all 48 graphs.
GRAPHS_PER_BATCH = 0
ACCELERATOR = "cuda:0"
STRICT_AUDITS = False
RECLAIM_STALE_LOCK = False

REPO_URL = "https://github.com/joshgreenwa/Graph-Specialisation-and-Metrics.git"
REPO_REVISION = "expansion/carriage_experiments"
REPO_DIR = "/content/Graph-Specialisation-and-Metrics"
GITHUB_SECRET = "dissertation_key"


In [ ]:
# Mount Drive and check out the requested branch/commit without exposing the token.
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
token = userdata.get(GITHUB_SECRET) or os.environ.get(GITHUB_SECRET)
suffix = REPO_URL.removeprefix("https://github.com/")
clone_url = (
    f"https://x-access-token:{quote(str(token).strip(), safe='')}@github.com/{suffix}"
    if token else REPO_URL
)
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", clone_url, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", clone_url], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
for entry in (str(repo), str(repo / "src")):
    if entry in sys.path:
        sys.path.remove(entry)
    sys.path.insert(0, entry)
for name in tuple(sys.modules):
    if name == "graph_specialisation_metrics" or name.startswith("graph_specialisation_metrics."):
        del sys.modules[name]


In [ ]:
# Execute setup, one isolated worker, a smoke/preflight, status, or finalization.
from experiments.methodology.zinc_qm9_canonical_colab_worker import run_frontend

result = run_frontend(
    mode=MODE,
    drive_folder=DRIVE_FOLDER,
    worker_index=WORKER_INDEX,
    task=(TASK or None),
    seed=(TRAIN_SEED if TASK else None),
    graphs_per_batch=(GRAPHS_PER_BATCH or None),
    accelerator=ACCELERATOR,
    strict_audits=STRICT_AUDITS,
    reclaim_stale_lock=RECLAIM_STALE_LOCK,
)
result
